> **Ekip paylaşım kopyası:** Kayıtlı çıktılar ve widget durumu dosya boyutunu azaltmak için temizlenmiştir; analiz hücreleri korunur. Veri dosyası Git deposuna dahil değildir. Kurulum, veri yolu ve mevcut veri dosyasını kopyalamadan kullanma seçenekleri için `../README.md` dosyasına bakın. `.venv` kernel'ini seçip Run All çalıştırın. Kaynak projedeki notebook ve çıktılar değiştirilmemiştir.

# DS02 — tek uçuş içinde sensör davranışları
Yalnızca **dev** verileri kullanılır. Başlangıç seçimi **Motor 2 / Uçuş 1**. 14 fiziksel ve 14 sanal sensör ayrı Y eksenlerinde gösterilir; alt, Mach, TRA ve T2 ayrıca incelenir.

X ekseni `sample_index`: uçuş içinde sıfırdan başlayan **örnek sırası**, gerçek saniye değildir. Artan HDF5 satır sırası korunur. Birimler metadata içinde doğrulanmadığı için tahmini fiziksel birimler yazılmaz; Y ekseni özgün veri değerini gösterir. Ölçekleme, filtreleme, yeniden örnekleme veya modelleme uygulanmaz.

VS Code'da `.venv` kernel'ini seçip Run All çalıştırın. Motor/uçuş seçtikten sonra **Uçuşu çiz** düğmesine basın. Büyük grafik için sensör seçimi yüklenmiş uçuşta geçerlidir.

### Öğretici okuma rehberi ve değişken sözlüğü
[DS02 Türkçe değişken sözlüğü](../docs/DS02_Degisken_Sozlugu.md), 47 özgün değişkenin tam İngilizce/Türkçe adlarını, birimlerini, bileşenlerini ve yorum sınırlarını açıklar. A kimlik/etiket, W koşul, X_s fiziksel ölçülebilir yanıt, X_v sanal model çıktısı, T simülasyon iç sağlığı, Y/RUL hedeftir. Fiziksel sensör adı bütün yanıtların gerçek bir arızalı motordan kaydedildiği anlamına gelmez; N-CMAPSS motor yanıtları simüle edilmiştir.

Makale Tables 4–8'e göre sıcaklık °R, basınç psia, hız rpm, kütlesel debi pps/lbm/s'dir. TRA birimi %; stall marjları kaynakta boyutsuz gösterilir. Grafik kodu değiştirilmediği için “özgün değer” etiketlerinin fiziksel birimleri sözlükten okunmalıdır. Önceki birim-doğrulama notlarının üzerine bu yeni kaynak açıklaması eklenmiştir; hesaplamalar değişmemiştir.


In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import ipywidgets as widgets
import os
import sys
# Matplotlib ?nbelle?i proje sanal ortam?nda tutulur.
os.environ.setdefault("MPLCONFIGDIR", str(Path(__import__("tempfile").gettempdir()) / "n_cmapss_matplotlib_cache"))
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Paylaşım kopyası: veri yolu bilgisayardan bağımsız çözülür.
import os

def resolve_ncmapss_paths(start=None):
    working_directory = Path.cwd().resolve() if start is None else Path(start).resolve()
    project_root = None
    for parent in [working_directory, *working_directory.parents]:
        if parent.name == "n_cmapss" and (parent / "notebooks").is_dir():
            project_root = parent
            break
        candidate = parent / "n_cmapss"
        if (candidate / "notebooks").is_dir():
            project_root = candidate
            break
    if project_root is None:
        raise FileNotFoundError("Notebook'u ortak repo, n_cmapss veya n_cmapss/notebooks klasöründen çalıştırın.")
    external_path = os.environ.get("N_CMAPSS_H5")
    h5_path = Path(external_path).expanduser().resolve() if external_path else project_root / "data" / "N-CMAPSS_DS02-006.h5"
    return project_root, h5_path

project_root, h5_path = resolve_ncmapss_paths()
if not h5_path.is_file():
    raise FileNotFoundError(
        f"Veri dosyası bulunamadı: {h5_path}. HDF5 repoya dahil değildir. "
        "Dosyayı yerel n_cmapss/data/ altında sağlayın veya kernel ortamında "
        "N_CMAPSS_H5 değişkenini mevcut dosyanın tam yoluna ayarlayın. README'ye bakın."
    )

DEV_KEYS = ["A_dev", "W_dev", "X_s_dev", "X_v_dev", "T_dev", "Y_dev"]
VAR_KEYS = ["A_var", "W_var", "X_s_var", "X_v_var", "T_var"]
CHUNK_ROWS = 100_000

def decode_names(dataset):
    return [v.decode("utf-8").rstrip("\x00") if isinstance(v, (bytes, np.bytes_)) else str(v)
            for v in np.asarray(dataset[()]).reshape(-1)]

with h5py.File(h5_path, "r") as h5:
    names = {key: decode_names(h5[key]) for key in VAR_KEYS}
    column_names = {key: names[var] for key, var in zip(DEV_KEYS[:-1], VAR_KEYS)}
    column_names["Y_dev"] = ["RUL"]
    row_counts = {key: h5[key].shape[0] for key in DEV_KEYS}
    if len(set(row_counts.values())) != 1:
        raise ValueError(f"Satır sayıları eşit değil: {row_counts}")
    for key in DEV_KEYS:
        if h5[key].ndim != 2 or h5[key].shape[1] != len(column_names[key]):
            raise ValueError(f"Sütun metadata uyumsuzluğu: {key}")
all_columns = [name for key in DEV_KEYS for name in column_names[key]]
assert len(all_columns) == len(set(all_columns)) == 47
unit_col = names["A_var"].index("unit")
cycle_col = names["A_var"].index("cycle")
print("dev satır sayısı:", row_counts["A_dev"])

%matplotlib inline
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.grid": True, "grid.alpha": 0.25})


In [ ]:
# Grafik etiketleri için tek merkezi başvuru; sayısal veriye dönüşüm uygulanmaz.
import json as _ds02_json
from textwrap import fill as _ds02_wrap
_ds02_metadata_path = project_root / "docs" / "assets" / "ds02_variable_metadata.json"
with _ds02_metadata_path.open(encoding="utf-8") as _ds02_stream:
    _ds02_metadata = _ds02_json.load(_ds02_stream)
DS02_VARIABLES = _ds02_metadata["variables"]
AXIS_LABELS = _ds02_metadata["axis_labels"]

def variable_unit(name):
    return DS02_VARIABLES[name]["compact_unit"]

def variable_title(name):
    return f"{name} ({variable_unit(name)})"

def variable_label(name):
    entry = DS02_VARIABLES[name]
    return _ds02_wrap(f"{name} — {entry['turkish']} ({entry['unit']})", width=70)


> [Motor ve Sensör Atlası](../docs/DS02_Motor_ve_Sensor_Atlasi.md) ve [merkezi birim sözlüğü](../docs/assets/ds02_variable_metadata.json) bu notebook'un etiket başvurusudur. Hesaplama/veri yolu mantığı korunmuştur.


## Motor/uçuş kataloğu
`A_dev` içindeki unit/cycle sütunları 100.000 satırlık parçalarla taranır. Yalnızca motor/uçuş çiftleri ve eşleşen satır aralıkları saklanır. Tüm dev verisi pandas'a yüklenmez.

In [ ]:
flight_ranges = {}
with h5py.File(h5_path, "r") as h5:
    a = h5["A_dev"]
    for start in range(0, a.shape[0], CHUNK_ROWS):
        stop = min(start + CHUNK_ROWS, a.shape[0])
        units = a[start:stop, unit_col]
        cycles = a[start:stop, cycle_col]
        if not (np.isfinite(units).all() and np.isfinite(cycles).all()
                and np.equal(units, np.floor(units)).all()
                and np.equal(cycles, np.floor(cycles)).all()):
            raise ValueError("Motor/uçuş kimlikleri sonlu tam sayı olmalıdır")
        changes = np.flatnonzero((units[1:] != units[:-1]) | (cycles[1:] != cycles[:-1])) + 1
        boundaries = np.r_[0, changes, len(units)]
        for left, right in zip(boundaries[:-1], boundaries[1:]):
            key = (int(units[left]), int(cycles[left]))
            begin, end = start + int(left), start + int(right)
            ranges = flight_ranges.setdefault(key, [])
            if ranges and ranges[-1][1] == begin:
                ranges[-1] = (ranges[-1][0], end)
            else:
                ranges.append((begin, end))
engine_flights = {}
for unit, cycle in sorted(flight_ranges):
    engine_flights.setdefault(unit, []).append(cycle)
assert sum(end - begin for ranges in flight_ranges.values() for begin, end in ranges) == row_counts["A_dev"]
catalog = pd.DataFrame([
    {"unit": unit, "uçuş_sayısı": len(cycles), "ilk_cycle": min(cycles), "son_cycle": max(cycles)}
    for unit, cycles in engine_flights.items()
])
display(catalog)
print("Toplam motor/uçuş çifti:", len(flight_ranges))


## Seçilen uçuşu okuma
Aynı satır aralıklarından sadece A, W, X_s ve X_v okunur. Bozulma parametreleri ve RUL grafikleri bu aşamanın kapsamına alınmaz. Sensör isimleri doğrudan `X_s_var` ve `X_v_var` metadata dizilerinden gelir. Bütün seçili uçuş örnekleri kullanılır.

### 28 sensörün fiziksel aileleri
X_s: sıcaklık T24/T30/T48/T50; basınç P15/P2/P21/P24/Ps30/P40/P50; mil hızları Nf/Nc; yakıt debisi Wf. X_v: iç sıcaklık/basınç, akışlar, stall marjları, phi gibi model çıktıları. W31/W32 için makale yalnızca HPT soğutma bleed akışını belirtir; rotor/stator kademesi varsayılmamıştır. [Sensör sözlüğü](../docs/DS02_Degisken_Sozlugu.md#xs-fiziksel).

In [ ]:
physical_sensors = names["X_s_var"]
virtual_sensors = names["X_v_var"]
sensors = physical_sensors + virtual_sensors
conditions = ["alt", "Mach", "TRA", "T2"]
assert len(physical_sensors) == len(virtual_sensors) == 14
assert set(conditions).issubset(names["W_var"])

def load_sensor_flight(unit, cycle):
    key = (int(unit), int(cycle))
    if key not in flight_ranges:
        raise ValueError(f"Motor/uçuş bulunamadı: {key}")
    frames = []
    with h5py.File(h5_path, "r") as h5:
        for name in ["A_dev", "W_dev", "X_s_dev", "X_v_dev"]:
            blocks = [h5[name][begin:end, :] for begin, end in flight_ranges[key]]
            values = blocks[0] if len(blocks) == 1 else np.concatenate(blocks, axis=0)
            frames.append(pd.DataFrame(values, columns=column_names[name]))
    table = pd.concat(frames, axis=1, verify_integrity=True)
    table.insert(0, "sample_index", np.arange(len(table), dtype=np.int64))
    assert (table["unit"] == key[0]).all() and (table["cycle"] == key[1]).all()
    return table

X_LABEL = AXIS_LABELS["sample_index"]
def label_axis(ax, table, name, color, compact=False):
    ax.plot(table["sample_index"], table[name], linewidth=0.7, color=color)
    ax.set_title(variable_title(name), fontsize=11)
    ax.set_ylabel(variable_unit(name) if compact else variable_label(name), fontsize=9)
    ax.tick_params(labelsize=8)

def sensor_overview(table, key):
    # X ortak, Y bağımsızdır: farklı birimlerin aynı eksende karşılaştırılması yapılmaz.
    fig, axes = plt.subplots(7, 4, figsize=(18, 22), sharex=True, sharey=False, layout="constrained")
    for ax, name in zip(axes.flat, sensors):
        label_axis(ax, table, name, "tab:blue" if name in physical_sensors else "tab:orange", compact=True)
        ax.set_title(f"{variable_title(name)} — {'fiziksel' if name in physical_sensors else 'sanal'}", fontsize=11)
    fig.suptitle(f"Motor {key[0]} / Uçuş {key[1]} — 28 sensör genel bakışı", fontsize=16)
    fig.supxlabel(X_LABEL, fontsize=12)
    return fig

def detail_plot(table, key, name):
    if name not in sensors:
        raise ValueError(f"Bilinmeyen sensör: {name}")
    fig, ax = plt.subplots(figsize=(14, 5), layout="constrained")
    label_axis(ax, table, name, "tab:blue" if name in physical_sensors else "tab:orange", compact=False)
    ax.set_title(f"Motor {key[0]} / Uçuş {key[1]} — {variable_title(name)}: ayrıntılı görünüm")
    ax.set_xlabel(X_LABEL)
    return fig

def condition_plots(table, key):
    fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True, sharey=False, layout="constrained")
    for ax, name in zip(axes, conditions):
        label_axis(ax, table, name, "tab:green")
    fig.suptitle(f"Motor {key[0]} / Uçuş {key[1]} — uçuş koşulları", fontsize=15)
    axes[-1].set_xlabel(X_LABEL)
    return fig

def show_figure(fig):
    # Seçimler arasında açık matplotlib figürlerinin birikmesini önle.
    try:
        display(fig)
    finally:
        plt.close(fig)


## İlk test ve kaydedilmiş grafikler: Motor 2 / Uçuş 1
28 sensör ve dört koşulun kaynak verilerle eşleşmesi, örnek sırası ve satır sayısı kontrol edilir. Genel bakış bağımsız ölçekli 28 panel içerir. Ayrıntılı grafiğin başlangıç sensörü ilk fiziksel sensördür.

### Motor atlası ve grafik/tablo birimleri
[Motor ve Sensör Atlası](../docs/DS02_Motor_ve_Sensor_Atlasi.md) bölge eşleştirmelerini, özgün kesit şemasını ve artış/azalışta araştırılabilecek olasılıkları açıklar. [47 değişken sözlüğü](../docs/DS02_Degisken_Sozlugu.md) korunmuştur. Birimler [merkezi JSON etiket sözlüğünden](../docs/assets/ds02_variable_metadata.json) alınır; veri dönüştürülmez, özgün sütun adları değişmez.

**TRA: % (Table 4), derece değil.** phi küçük başlıklarda pps/psi* ile gösterilir: yıldız kaynak biriminin korunduğunu ama yerel ölçeğin doğrulanmadığını belirtir. Stall margin ve sağlık parametreleri boyutsuzdur; yüzde sağlık/verim kaybı değildir. Medyan/Q1–Q3 ve min–maksimum bantlarının önceki anlamları korunur. sample_index ölçüm sırası (saniye değil), cycle uçuş döngüsü, RUL kalan uçuş döngüsüdür. Normalize yaşam 0–1, boyutsuz ve yalnızca görsel karşılaştırmadır.

<details><summary>Değişken adları, fiziksel anlamları ve birimleri</summary>

| Kod | Türkçe açıklama | Birim |
|---|---|---|
| `alt` | Uçuş irtifası | ft |
| `Mach` | Uçuş Mach sayısı | boyutsuz |
| `TRA` | Gaz kolu / güç talebi göstergesi | % |
| `T2` | Fan giriş toplam sıcaklığı | °R |
| `T24` | LPC çıkış toplam sıcaklığı | °R |
| `T30` | HPC çıkış toplam sıcaklığı | °R |
| `T48` | HPT çıkış toplam sıcaklığı | °R |
| `T50` | LPT çıkış toplam sıcaklığı | °R |
| `P15` | Bypass kanalı toplam basıncı | psia |
| `P2` | Fan giriş toplam basıncı | psia |
| `P21` | Fan çıkış toplam basıncı | psia |
| `P24` | LPC çıkış toplam basıncı | psia |
| `Ps30` | HPC çıkış statik basıncı | psia |
| `P40` | Yanma odası çıkışı / HPT girişi toplam basıncı | psia |
| `P50` | LPT çıkış toplam basıncı | psia |
| `Nf` | Fanın fiziksel dönüş hızı | rpm |
| `Nc` | Çekirdeğin fiziksel dönüş hızı | rpm |
| `Wf` | Yakıt kütlesel debisi | lbm/s |
| `T40` | Yanma odası çıkış toplam sıcaklığı | °R |
| `P30` | HPC çıkış toplam basıncı | psia |
| `P45` | HPT çıkış toplam basıncı | psia |
| `W21` | Fan kütlesel debisi | lbm/s |
| `W22` | LPC çıkış kütlesel debisi | lbm/s |
| `W25` | HPC giriş kütlesel debisi | lbm/s |
| `W31` | HPT soğutma bleed debisi, W31 | lbm/s |
| `W32` | HPT soğutma bleed debisi, W32 | lbm/s |
| `W48` | HPT çıkış kütlesel debisi | lbm/s |
| `W50` | LPT çıkış kütlesel debisi | lbm/s |
| `SmFan` | Fan stall marjı | boyutsuz |
| `SmLPC` | LPC stall marjı | boyutsuz |
| `SmHPC` | HPC stall marjı | boyutsuz |
| `phi` | Yakıt debisinin Ps30'a oranı (makale tanımı) | pps/psi; dosya ölçeği belirsiz |

</details>

**Kayıtlı çıktı notu:** Kaynak notebook'lardaki başlangıç grafiklerinin görselleri, çizgi/bant verilerinin eşitliği doğrulanarak mevcut CSV özetleri ve tek uçuş verisiyle yeni etiketlerle yeniden çizildi. Veri içeriği değiştirilmedi. Paylaşım kopyalarının temiz çıktı durumu korunur. Tam EDA yeniden çalıştırılmadı.


### Uçuş içi grafik yaşam döngüsü grafiği değildir
X ekseni örnek sırasıdır: 1000, 1000 saniye demek değildir. Her Y ekseni kendi fiziksel büyüklüğünü taşır. °R/°C, psia/psig farklıdır; stall margin sağlık yüzdesi değildir. Bu eğriler bir uçuşun koşul ve motor yanıtını birlikte gösterir; bütün motor ömrü boyunca bozulma eğrisi değildir. [Birim ve fizik rehberi](../docs/DS02_Degisken_Sozlugu.md#motor-ve-fizik).

In [ ]:
initial_key = (2, 1)
initial_table = load_sensor_flight(*initial_key)
expected_rows = sum(end-begin for begin, end in flight_ranges[initial_key])
assert len(initial_table) == expected_rows
assert initial_table.shape[1] == 37  # sample_index + A(4) + W(4) + X_s(14) + X_v(14)
assert np.array_equal(initial_table["sample_index"], np.arange(expected_rows))
with h5py.File(h5_path, "r") as h5:
    offset = 0
    for begin, end in flight_ranges[initial_key]:
        for start in range(begin, end, CHUNK_ROWS):
            stop = min(start+CHUNK_ROWS, end)
            count = stop-start
            for key in ["A_dev", "W_dev", "X_s_dev", "X_v_dev"]:
                np.testing.assert_array_equal(initial_table.iloc[offset:offset+count][column_names[key]].to_numpy(), h5[key][start:stop, :])
            offset += count
print(f"Motor 2 / Uçuş 1: {len(initial_table):,} örnek; 28 sensör; 4 uçuş koşulu.")
print(f"sample_index: {initial_table.sample_index.min()}–{initial_table.sample_index.max()} (saniye değil)")
print("Kaynak verilerle eşleşme başarılı. Eksik sensör değeri sayısı:", int(initial_table[sensors].isna().sum().sum()))
show_figure(sensor_overview(initial_table, initial_key))
show_figure(detail_plot(initial_table, initial_key, physical_sensors[0]))
show_figure(condition_plots(initial_table, initial_key))


## Etkileşimli sensör inceleyici
Aşağıdaki kontrollerle yeni uçuşu çizin. Motor/uçuş değişince eski grafikler temizlenir. Sensör seçimi sadece büyük grafiği yeniler; uçuşu tekrar okumaz. Genel bakışta bütün 28 sensör ve koşullarda dört ayrı Y ekseni korunur. Başlangıç grafikleri yukarıda kaydedilmiştir; etkileşimli alanda görmek için **Uçuşu çiz** düğmesine basın.

### Koşullarla birlikte yorumlama
alt ft, Mach boyutsuz, TRA %, T2 °R'dir. Sensör yükselirken TRA da artıyorsa güç talebi olası etkidir; eşzamanlılık nedensellik kanıtı değildir. Aynı uçuşta hs/RUL sabitken sensörler değişebilir. T2 fan girişi koşulu, T24 LPC çıkışı yanıtıdır. [W sözlüğü](../docs/DS02_Degisken_Sozlugu.md#w-kosullar).

In [ ]:
engine = widgets.Dropdown(options=list(engine_flights), value=2, description="Motor:")
flight = widgets.Dropdown(options=engine_flights[2], value=1, description="Uçuş:")
sensor = widgets.Dropdown(options=[(f"{variable_title(n)} — fiziksel", n) for n in physical_sensors] + [(f"{variable_title(n)} — sanal", n) for n in virtual_sensors], description="Sensör:")
plot_button = widgets.Button(description="Uçuşu çiz", button_style="primary")
status = widgets.HTML(value="Başlangıç grafikleri yukarıda: Motor 2 / Uçuş 1.")
overview_output, detail_output, conditions_output = widgets.Output(), widgets.Output(), widgets.Output()
state = {"table": initial_table, "key": initial_key}

def clear_plots():
    for output in [overview_output, detail_output, conditions_output]:
        with output:
            clear_output(wait=False)

def invalidate():
    state.update(table=None, key=None)
    status.value = "Yeni seçimi yüklemek için Uçuşu çiz düğmesine basın."
    clear_plots()

def change_engine(change):
    flight.options = engine_flights[change["new"]]
    flight.value = engine_flights[change["new"]][0]
    invalidate()

def redraw_detail(change=None):
    if state["table"] is None:
        return
    with detail_output:
        clear_output(wait=True)
        show_figure(detail_plot(state["table"], state["key"], sensor.value))

def on_plot(button):
    plot_button.disabled = True
    clear_plots()
    try:
        table = load_sensor_flight(engine.value, flight.value)
        key = (engine.value, flight.value)
        state.update(table=table, key=key)
        status.value = f"Motor {key[0]} / Uçuş {key[1]} — {len(table):,} örnek. X ekseni örnek sırası; saniye değil."
        with overview_output:
            show_figure(sensor_overview(table, key))
        redraw_detail()
        with conditions_output:
            show_figure(condition_plots(table, key))
    except Exception as exc:
        invalidate()
        status.value = "Çizim başarısız; hata ayrıntısı aşağıda."
        with detail_output:
            print(f"Hata: {exc}")
    finally:
        plot_button.disabled = False

engine.observe(change_engine, names="value")
flight.observe(lambda change: invalidate(), names="value")
sensor.observe(redraw_detail, names="value")
plot_button.on_click(on_plot)
display(widgets.VBox([widgets.HBox([engine, flight, plot_button]), sensor, status,
    widgets.HTML("<h3>28 sensör genel bakışı</h3>"), overview_output,
    widgets.HTML("<h3>Seçilen sensörün ayrıntısı</h3>"), detail_output,
    widgets.HTML("<h3>Uçuş koşulları</h3>"), conditions_output]))
